# Team 22 – Distributed Sorting with MPI
**Real-world benchmark: UCI Online Retail | C++17 | OpenMPI | Google Colab**

This notebook evaluates **Sample Sort** and **Parallel Bitonic Sort** using:
- Controlled synthetic inputs (uniform, Gaussian, sorted, reverse and duplicates).
- A real-world dataset: **UCI Online Retail**, using `UnitPrice` converted to integer pence.
- Multiple MPI process counts and splitter strategies.
- Correctness validation, execution time, communication time, throughput and load balance.

**How to present**
1. Explain the problem and why sorting large data can benefit from parallel processing.
2. Introduce the UCI Online Retail dataset and explain that `UnitPrice` is the sorting key.
3. Explain how MPI distributes records among processes.
4. Compare Sample Sort and Bitonic Sort using the same dataset and process count.
5. Interpret the graphs and discuss communication overhead and correctness.

**Run from top to bottom.** The C++ source is created inside this notebook, so there is no separate source ZIP upload step. Colab's available CPU count is limited; `--oversubscribe` allows MPI ranks to launch but does not create extra physical cores.


**Problem Statement**

In conventional sorting systems, processing large datasets using a single processor can lead to increased execution time and inefficient utilization of computational resources. As the volume of data increases, the need for faster and more efficient sorting becomes important.

The objective of this project is to design and implement a Distributed Sorting System using the Message Passing Interface (MPI) that distributes sorting operations across multiple processes. The system implements Sample Sort and Bitonic Sort to investigate parallel sorting performance.

The project aims to analyse how different input distributions, splitter selection strategies, and numbers of MPI processes influence sorting correctness, execution time, communication overhead, throughput, and workload balance.

The system also provides experimental graphs and performance measurements to understand the advantages and limitations of distributed sorting compared with single-process execution.

**Key problems addressed**
1. Reducing sorting execution time through parallel processing.
2. Distributing data efficiently among multiple MPI processes.
3. Minimizing communication overhead during data exchange.
4. Maintaining balanced workloads across processes.
5. Evaluating sorting performance under different input distributions.
6. Comparing the efficiency of Sample Sort and Bitonic Sort.

**Main objective**

To develop and experimentally evaluate an MPI-based distributed sorting system that demonstrates how parallel algorithms, communication, and workload distribution affect sorting performance and scalability.

## 1. Install OpenMPI and compiler

In [ ]:
!apt-get update -qq
!apt-get install -y -qq openmpi-bin libopenmpi-dev g++

## 2. Verify installation

In [ ]:
import os, subprocess
print("CPU cores visible to Colab:", os.cpu_count())
subprocess.run(["mpic++", "--version"], check=True)
subprocess.run(["mpirun", "--version"], check=True)

## 3. Download the real-world dataset — UCI Online Retail

We use the official UCI Online Retail dataset. It contains transaction records from an online retailer.

**Sorting key:** `UnitPrice` (in pounds sterling). We convert it to integer pence (`UnitPrice × 100`) because the MPI implementation sorts integer keys. This avoids floating-point comparison issues.

Official dataset: [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail)


**Scope clarification:** The MPI program sorts the extracted `UnitPrice` numeric keys. It does not rearrange every column of the original retail transaction rows. Each key receives a unique ID so the program can verify that no key-record pair is lost or duplicated.


In [ ]:
import os
import urllib.request

DATA_DIR = "/content/uci_online_retail"
os.makedirs(DATA_DIR, exist_ok=True)

DATA_URL = "https://archive.ics.uci.edu/static/public/352/online+retail.zip"
ZIP_PATH = os.path.join(DATA_DIR, "online_retail.zip")

if not os.path.exists(ZIP_PATH):
    print("Downloading UCI Online Retail dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

print("Dataset archive ready:", ZIP_PATH)
print("File size (MB):", round(os.path.getsize(ZIP_PATH) / (1024 * 1024), 2))


## 4. Extract and inspect the dataset

The archive contains the Excel data file. We load it with pandas and inspect its structure before preparing the sorting keys.


In [ ]:
import zipfile
import glob
import pandas as pd
import numpy as np

try:
    import openpyxl
except ImportError:
    !pip -q install openpyxl
    import openpyxl

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    archive.extractall(DATA_DIR)

excel_files = glob.glob(os.path.join(DATA_DIR, "**", "*.xlsx"), recursive=True)
if not excel_files:
    raise FileNotFoundError("The UCI archive did not contain an .xlsx file.")

RETAIL_XLSX = excel_files[0]
retail_df = pd.read_excel(RETAIL_XLSX)

print("Dataset loaded:", RETAIL_XLSX)
print("Rows:", len(retail_df))
print("Columns:", list(retail_df.columns))
display(retail_df.head())


## 5. Prepare the MPI C++ implementation

The C++ program retains the original synthetic-input mode and adds a `--file` mode. In file mode, rank 0 reads integer keys from a text file, assigns each record a unique ID, and distributes records with MPI.

The unique ID allows the program to verify both sorted order and preservation of every input record, including duplicate prices.


In [ ]:
CPP_SOURCE = '#include <mpi.h>\n#include <algorithm>\n#include <cmath>\n#include <cstdint>\n#include <climits>\n#include <iostream>\n#include <iterator>\n#include <numeric>\n#include <random>\n#include <string>\n#include <vector>\n#include <limits>\n#include <fstream>\n\nstruct Record {\n    int value;\n    long long id;\n};\n\nstruct RecordLess {\n    bool operator()(const Record& a, const Record& b) const {\n        if (a.value != b.value) return a.value < b.value;\n        return a.id < b.id;\n    }\n};\n\nstatic MPI_Datatype make_record_type() {\n    MPI_Datatype type;\n    Record sample{};\n    int lengths[2] = {1, 1};\n    MPI_Aint offsets[2];\n    MPI_Datatype types[2] = {MPI_INT, MPI_LONG_LONG_INT};\n    MPI_Get_address(&sample.value, &offsets[0]);\n    MPI_Get_address(&sample.id, &offsets[1]);\n    offsets[1] -= offsets[0];\n    offsets[0] = 0;\n    MPI_Type_create_struct(2, lengths, offsets, types, &type);\n    MPI_Type_commit(&type);\n    return type;\n}\n\nstatic std::vector<Record> generate_input(int n, const std::string& distribution) {\n    std::vector<Record> data(n);\n    std::mt19937_64 rng(1234567);\n    if (distribution == "gaussian") {\n        std::normal_distribution<double> dist(0.0, 100000.0);\n        for (int i = 0; i < n; ++i) {\n            double x = std::max(-1000000.0, std::min(1000000.0, dist(rng)));\n            data[i] = {static_cast<int>(std::llround(x)), i};\n        }\n    } else if (distribution == "duplicates") {\n        std::uniform_int_distribution<int> dist(0, 9);\n        for (int i = 0; i < n; ++i) data[i] = {dist(rng), i};\n    } else {\n        std::uniform_int_distribution<int> dist(-1000000, 1000000);\n        for (int i = 0; i < n; ++i) data[i] = {dist(rng), i};\n        if (distribution == "sorted")\n            std::sort(data.begin(), data.end(), RecordLess{});\n        else if (distribution == "reverse") {\n            std::sort(data.begin(), data.end(), RecordLess{});\n            std::reverse(data.begin(), data.end());\n        }\n    }\n    return data;\n}\n\nstatic void distribute_input(const std::vector<Record>& input, std::vector<Record>& local,\n                             int n, int rank, int p, MPI_Datatype record_type) {\n    std::vector<int> counts(p), displs(p);\n    int base = n / p, rem = n % p, offset = 0;\n    for (int r = 0; r < p; ++r) {\n        counts[r] = base + (r < rem ? 1 : 0);\n        displs[r] = offset;\n        offset += counts[r];\n    }\n    local.resize(counts[rank]);\n    MPI_Scatterv(rank == 0 ? input.data() : nullptr, counts.data(), displs.data(), record_type,\n                 local.data(), counts[rank], record_type, 0, MPI_COMM_WORLD);\n}\n\nstatic std::vector<Record> choose_samples(const std::vector<Record>& local,\n                                          const std::string& strategy, int p, int oversample,\n                                          int rank) {\n    std::vector<Record> samples;\n    if (p <= 1 || local.empty()) return samples;\n    int wanted = (strategy == "oversample") ? oversample * (p - 1) : (p - 1);\n    int count = std::min<int>(wanted, local.size());\n    samples.reserve(count);\n    if (strategy == "random") {\n        std::mt19937 rng(98765 + rank);\n        std::uniform_int_distribution<int> pick(0, static_cast<int>(local.size()) - 1);\n        for (int i = 0; i < count; ++i) samples.push_back(local[pick(rng)]);\n    } else {\n        for (int i = 0; i < count; ++i) {\n            size_t index = std::min(local.size() - 1,\n                static_cast<size_t>((static_cast<double>(i) + 0.5) * local.size() / count));\n            samples.push_back(local[index]);\n        }\n    }\n    return samples;\n}\n\nstatic std::vector<Record> sample_sort(std::vector<Record>& local, int p, int rank,\n                                       MPI_Datatype record_type, const std::string& strategy,\n                                       int oversample, double& local_sort_time,\n                                       double& communication_time, double& merge_time) {\n    double t = MPI_Wtime();\n    std::sort(local.begin(), local.end(), RecordLess{});\n    local_sort_time += MPI_Wtime() - t;\n\n    t = MPI_Wtime();\n    std::vector<Record> samples = choose_samples(local, strategy, p, oversample, rank);\n    int sample_count = static_cast<int>(samples.size());\n    std::vector<int> sample_counts(p), sample_displs(p);\n    MPI_Allgather(&sample_count, 1, MPI_INT, sample_counts.data(), 1, MPI_INT, MPI_COMM_WORLD);\n    int total_samples = 0;\n    for (int r = 0; r < p; ++r) {\n        sample_displs[r] = total_samples;\n        total_samples += sample_counts[r];\n    }\n    std::vector<Record> all_samples(total_samples);\n    MPI_Allgatherv(samples.data(), sample_count, record_type, all_samples.data(),\n                   sample_counts.data(), sample_displs.data(), record_type, MPI_COMM_WORLD);\n    std::sort(all_samples.begin(), all_samples.end(), RecordLess{});\n\n    std::vector<Record> splitters;\n    if (p > 1 && !all_samples.empty()) {\n        for (int i = 1; i < p; ++i) {\n            size_t index = std::min(all_samples.size() - 1,\n                static_cast<size_t>((static_cast<double>(i) * all_samples.size()) / p));\n            splitters.push_back(all_samples[index]);\n        }\n    } else if (p > 1) {\n        splitters.assign(p - 1, Record{INT_MAX, LLONG_MAX});\n    }\n\n    std::vector<std::vector<Record>> buckets(p);\n    auto begin = local.begin();\n    for (int dest = 0; dest < p; ++dest) {\n        if (dest == p - 1) {\n            buckets[dest].assign(begin, local.end());\n        } else {\n            auto end = std::upper_bound(begin, local.end(), splitters[dest], RecordLess{});\n            buckets[dest].assign(begin, end);\n            begin = end;\n        }\n    }\n    std::vector<int> send_counts(p), send_displs(p), recv_counts(p), recv_displs(p);\n    int send_total = 0;\n    for (int r = 0; r < p; ++r) {\n        send_counts[r] = static_cast<int>(buckets[r].size());\n        send_displs[r] = send_total;\n        send_total += send_counts[r];\n    }\n    std::vector<Record> send_buffer;\n    send_buffer.reserve(send_total);\n    for (auto& b : buckets) send_buffer.insert(send_buffer.end(), b.begin(), b.end());\n    MPI_Alltoall(send_counts.data(), 1, MPI_INT, recv_counts.data(), 1, MPI_INT, MPI_COMM_WORLD);\n    int recv_total = 0;\n    for (int r = 0; r < p; ++r) {\n        recv_displs[r] = recv_total;\n        recv_total += recv_counts[r];\n    }\n    std::vector<Record> received(recv_total);\n    MPI_Alltoallv(send_buffer.data(), send_counts.data(), send_displs.data(), record_type,\n                  received.data(), recv_counts.data(), recv_displs.data(), record_type,\n                  MPI_COMM_WORLD);\n    communication_time += MPI_Wtime() - t;\n\n    t = MPI_Wtime();\n    std::sort(received.begin(), received.end(), RecordLess{});\n    merge_time += MPI_Wtime() - t;\n    return received;\n}\n\nstatic std::vector<Record> bitonic_sort(std::vector<Record>& local, int n, int p, int rank,\n                                        MPI_Datatype record_type, double& local_sort_time,\n                                        double& communication_time, double& merge_time) {\n    int chunk = (n + p - 1) / p;\n    local.resize(chunk, Record{INT_MAX, LLONG_MAX});\n    double t = MPI_Wtime();\n    std::sort(local.begin(), local.end(), RecordLess{});\n    local_sort_time += MPI_Wtime() - t;\n    std::vector<Record> partner_data(chunk);\n\n    for (int k = 2; k <= p; k <<= 1) {\n        for (int j = k >> 1; j > 0; j >>= 1) {\n            int partner = rank ^ j;\n            t = MPI_Wtime();\n            MPI_Sendrecv(local.data(), chunk, record_type, partner, 100,\n                         partner_data.data(), chunk, record_type, partner, 100,\n                         MPI_COMM_WORLD, MPI_STATUS_IGNORE);\n            communication_time += MPI_Wtime() - t;\n\n            t = MPI_Wtime();\n            std::vector<Record> combined;\n            combined.reserve(2 * chunk);\n            std::merge(local.begin(), local.end(), partner_data.begin(), partner_data.end(),\n                       std::back_inserter(combined), RecordLess{});\n            bool keep_low = ((rank & k) == 0) == ((rank & j) == 0);\n            if (keep_low)\n                std::copy(combined.begin(), combined.begin() + chunk, local.begin());\n            else\n                std::copy(combined.end() - chunk, combined.end(), local.begin());\n            merge_time += MPI_Wtime() - t;\n        }\n    }\n    local.erase(std::remove_if(local.begin(), local.end(),\n                               [](const Record& x) { return x.id == LLONG_MAX; }), local.end());\n    return local;\n}\n\nstatic bool gather_and_verify(const std::vector<Record>& local, const std::vector<Record>& original,\n                              int n, int rank, int p, MPI_Datatype record_type,\n                              double& communication_time) {\n    double t = MPI_Wtime();\n    int local_n = static_cast<int>(local.size());\n    std::vector<int> counts(p), displs(p);\n    MPI_Gather(&local_n, 1, MPI_INT, counts.data(), 1, MPI_INT, 0, MPI_COMM_WORLD);\n    int total = 0;\n    if (rank == 0) {\n        for (int r = 0; r < p; ++r) { displs[r] = total; total += counts[r]; }\n    }\n    std::vector<Record> all(rank == 0 ? total : 0);\n    MPI_Gatherv(local.data(), local_n, record_type, all.data(), counts.data(), displs.data(),\n                record_type, 0, MPI_COMM_WORLD);\n    communication_time += MPI_Wtime() - t;\n\n    int ok = 1;\n    if (rank == 0) {\n        if (static_cast<int>(all.size()) != n) ok = 0;\n        else {\n            for (size_t i = 1; i < all.size(); ++i)\n                if (RecordLess{}(all[i], all[i - 1])) { ok = 0; break; }\n            std::vector<unsigned char> seen(n, 0);\n            if (ok) for (const auto& x : all) {\n                if (x.id < 0 || x.id >= n || seen[x.id] || original[x.id].value != x.value) {\n                    ok = 0; break;\n                }\n                seen[x.id] = 1;\n            }\n        }\n    }\n    MPI_Bcast(&ok, 1, MPI_INT, 0, MPI_COMM_WORLD);\n    return ok == 1;\n}\n\nint main(int argc, char** argv) {\n    MPI_Init(&argc, &argv);\n    int rank, p;\n    MPI_Comm_rank(MPI_COMM_WORLD, &rank);\n    MPI_Comm_size(MPI_COMM_WORLD, &p);\n    MPI_Datatype record_type = make_record_type();\n\n    int n = 1000000, oversample = 4;\n    std::string algorithm = "sample", strategy = "oversample", distribution = "uniform";\n    bool file_mode = (argc > 1 && std::string(argv[1]) == "--file");\n    std::string input_path;\n\n    if (file_mode) {\n        if (argc < 5) {\n            if (rank == 0) std::cerr << "Usage: mpirun -np P ./team22_sort --file INPUT.txt sample|bitonic regular|random|oversample [oversample_factor]\\\\n";\n            MPI_Type_free(&record_type);\n            MPI_Finalize();\n            return 1;\n        }\n        input_path = argv[2];\n        algorithm = argv[3];\n        strategy = argv[4];\n        if (argc > 5) oversample = std::max(1, std::stoi(argv[5]));\n    } else {\n        if (argc > 1) n = std::stoi(argv[1]);\n        if (argc > 2) algorithm = argv[2];\n        if (argc > 3) strategy = argv[3];\n        if (argc > 4) distribution = argv[4];\n        if (argc > 5) oversample = std::max(1, std::stoi(argv[5]));\n    }\n\n    std::vector<Record> original;\n    int input_ok = 1;\n    if (file_mode) {\n        if (rank == 0) {\n            std::ifstream in(input_path);\n            if (!in) input_ok = 0;\n            else {\n                long long value;\n                long long id = 0;\n                while (in >> value) {\n                    if (value < INT_MIN || value > INT_MAX) { input_ok = 0; break; }\n                    original.push_back({static_cast<int>(value), id++});\n                }\n                n = static_cast<int>(original.size());\n                if (in.bad() || n == 0) input_ok = 0;\n            }\n        }\n        MPI_Bcast(&input_ok, 1, MPI_INT, 0, MPI_COMM_WORLD);\n        MPI_Bcast(&n, 1, MPI_INT, 0, MPI_COMM_WORLD);\n        distribution = "uci_online_retail_unitprice";\n    }\n\n    int valid = (input_ok && n >= 0 && (algorithm == "sample" || algorithm == "bitonic") &&\n                 (strategy == "regular" || strategy == "random" || strategy == "oversample") &&\n                 (file_mode || distribution == "uniform" || distribution == "gaussian" ||\n                  distribution == "sorted" || distribution == "reverse" ||\n                  distribution == "duplicates"));\n    if (algorithm == "bitonic" && (p & (p - 1)) != 0) valid = 0;\n    if (n > INT_MAX / std::max(1, p)) valid = 0;\n    if (!valid) {\n        if (rank == 0) std::cerr << "Usage: mpirun -np P ./team22_sort N sample|bitonic regular|random|oversample uniform|gaussian|sorted|reverse|duplicates [oversample_factor]\\n"\n                                  << "Bitonic Sort requires P to be a power of two. N must fit MPI int counts.\\n";\n        MPI_Type_free(&record_type);\n        MPI_Finalize();\n        return 1;\n    }\n\n    if (!file_mode && rank == 0) original = generate_input(n, distribution);\n    std::vector<Record> local;\n    distribute_input(original, local, n, rank, p, record_type);\n\n    double local_sort_time = 0.0, communication_time = 0.0, merge_time = 0.0;\n    MPI_Barrier(MPI_COMM_WORLD);\n    double total_start = MPI_Wtime();\n    std::vector<Record> sorted;\n    if (algorithm == "sample")\n        sorted = sample_sort(local, p, rank, record_type, strategy, oversample,\n                             local_sort_time, communication_time, merge_time);\n    else\n        sorted = bitonic_sort(local, n, p, rank, record_type,\n                              local_sort_time, communication_time, merge_time);\n    MPI_Barrier(MPI_COMM_WORLD);\n    double total_time = MPI_Wtime() - total_start;\n\n    int local_output_count = static_cast<int>(sorted.size());\n    std::vector<int> output_counts(rank == 0 ? p : 0);\n    MPI_Gather(&local_output_count, 1, MPI_INT, output_counts.data(), 1, MPI_INT, 0, MPI_COMM_WORLD);\n    double load_imbalance = 0.0;\n    if (rank == 0 && n > 0) {\n        int largest = *std::max_element(output_counts.begin(), output_counts.end());\n        load_imbalance = static_cast<double>(largest) / (static_cast<double>(n) / p);\n    }\n\n    bool correct = gather_and_verify(sorted, original, n, rank, p, record_type, communication_time);\n    double max_total, max_local, max_comm, max_merge;\n    MPI_Reduce(&total_time, &max_total, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);\n    MPI_Reduce(&local_sort_time, &max_local, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);\n    MPI_Reduce(&communication_time, &max_comm, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);\n    MPI_Reduce(&merge_time, &max_merge, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);\n\n    if (rank == 0) {\n        std::cout << "Algorithm: " << algorithm << "\\nProcesses: " << p\n                  << "\\nInput keys: " << n << "\\nDistribution: " << distribution;\n        if (algorithm == "sample") std::cout << "\\nSplitter strategy: " << strategy;\n        std::cout << "\\nCorrectness: " << (correct ? "PASS" : "FAIL")\n                  << "\\nTotal time (max rank, s): " << max_total\n                  << "\\nLocal sort time (max rank, s): " << max_local\n                  << "\\nCommunication time (max rank, s): " << max_comm\n                  << "\\nMerge/final-sort time (max rank, s): " << max_merge\n                  << "\\nKeys/second: " << (max_total > 0 ? n / max_total : 0)\n                  << "\\nLoad imbalance (max rank keys / average keys): " << load_imbalance\n                  << "\\nPer-rank output counts: ";\n        for (int r = 0; r < p; ++r) std::cout << output_counts[r] << (r + 1 == p ? "\\n" : ", ");\n    }\n\n    MPI_Type_free(&record_type);\n    MPI_Finalize();\n    return correct ? 0 : 2;\n}\n'

CPP_FILE = "/content/team22_distributed_sort.cpp"
with open(CPP_FILE, "w", encoding="utf-8") as source_file:
    source_file.write(CPP_SOURCE)

print("MPI source created:", CPP_FILE)
print("Source size:", os.path.getsize(CPP_FILE), "bytes")


## 6. Compile the MPI C++ program

In [ ]:
import subprocess
import os

EXE = "/content/team22_sort"

compile_cmd = [
    "mpic++", "-O3", "-std=c++17",
    CPP_FILE, "-o", EXE
]

compiled = subprocess.run(
    compile_cmd,
    capture_output=True,
    text=True
)

if compiled.returncode == 0 and os.path.exists(EXE):
    print("Compilation successful!")
    print("Executable:", EXE)
else:
    print("Compilation failed!")
    print(compiled.stderr)

## 7. Prepare real-world numeric sorting keys

We select valid, non-negative `UnitPrice` values and convert pounds sterling to integer pence. The original row order is retained so the experiment tests sorting rather than pre-sorted data.

Each generated text file contains one integer key per line. The C++ program assigns a unique record ID while reading it.


In [ ]:
prices = pd.to_numeric(retail_df["UnitPrice"], errors="coerce").dropna()
prices = prices[prices >= 0]
retail_keys = (prices * 100).round().astype("int64").reset_index(drop=True)

print("Valid price records:", len(retail_keys))
print("Unique price keys:", retail_keys.nunique())
print("Minimum price (pence):", int(retail_keys.min()))
print("Maximum price (pence):", int(retail_keys.max()))
print("Duplicate-key records:", int(retail_keys.duplicated().sum()))


## 8. Explore the real dataset before sorting

These plots help explain the input data to your teacher. The price distribution is typically skewed, and repeated prices are expected in retail transactions.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(retail_keys / 100, bins=50)
axes[0].set_title("Distribution of UnitPrice")
axes[0].set_xlabel("Unit price (£)")
axes[0].set_ylabel("Number of records")

top_prices = retail_keys.value_counts().head(10).sort_values(ascending=True)
axes[1].barh([f"£{x/100:.2f}" for x in top_prices.index], top_prices.values)
axes[1].set_title("10 Most Frequent Unit Prices")
axes[1].set_xlabel("Record count")

plt.tight_layout()
plt.show()


## 9. Run MPI sorting on the UCI dataset

For a fair algorithm comparison, Sample Sort and Bitonic Sort receive the **same input file**, the same number of records and the same number of MPI processes.

The default experiment uses 4 processes. Bitonic Sort requires a power-of-two process count.


In [ ]:
import subprocess
import re
import os

def run_mpi_file(input_path, algorithm="sample", strategy="oversample",
                 processes=4, oversample_factor=4):
    command = [
        "mpirun", "--allow-run-as-root", "--oversubscribe",
        "-np", str(processes), EXE,
        "--file", input_path, algorithm, strategy, str(oversample_factor)
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("MPI stderr:", result.stderr)
    if result.returncode != 0:
        print("MPI returned non-zero exit code:", result.returncode)
    return result

def parse_file_metrics(result, algorithm, processes, n_records, strategy):
    output = result.stdout if result else ""

    def get(pattern):
        match = re.search(pattern, output)
        return float(match.group(1)) if match else np.nan

    correctness = re.search(r"Correctness:\s*(PASS|FAIL)", output)
    return {
        "Dataset": "UCI Online Retail",
        "Sorting Key": "UnitPrice (integer pence)",
        "Algorithm": algorithm,
        "Processes": processes,
        "Records": n_records,
        "Splitter": strategy if algorithm == "sample" else "-",
        "Correctness": correctness.group(1) if correctness else "ERROR",
        "Total Time (s)": get(r"Total time \(max rank, s\):\s*([0-9.eE+-]+)"),
        "Local Sort Time (s)": get(r"Local sort time \(max rank, s\):\s*([0-9.eE+-]+)"),
        "Communication Time (s)": get(r"Communication time \(max rank, s\):\s*([0-9.eE+-]+)"),
        "Merge Time (s)": get(r"Merge/final-sort time \(max rank, s\):\s*([0-9.eE+-]+)"),
        "Keys Per Second": get(r"Keys/second:\s*([0-9.eE+-]+)"),
        "Load Imbalance": get(r"Load imbalance \(max rank keys / average keys\):\s*([0-9.eE+-]+)")
    }


## 6. Run the first Sample Sort experiment

In [ ]:
def run_mpi(n=100000, algorithm="sample", strategy="oversample",
            distribution="uniform", processes=4, oversample_factor=4):
    cmd = [
        "mpirun", "--allow-run-as-root", "--oversubscribe",
        "-np", str(processes), EXE,
        str(n), algorithm, strategy, distribution, str(oversample_factor)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("MPI process returned non-zero exit code:", result.returncode)
    return result

first_run = run_mpi(
    n=100000, algorithm="sample", strategy="oversample",
    distribution="uniform", processes=4
)

## 7. Run Parallel Bitonic Sort

In [ ]:
bitonic_run = run_mpi(
    n=100000, algorithm="bitonic", strategy="regular",
    distribution="uniform", processes=4
)

## 8. Test all five input distributions with Sample Sort

In [ ]:
distributions = ["uniform", "gaussian", "sorted", "reverse", "duplicates"]
distribution_runs = {}

for dist in distributions:
    print("\\n" + "=" * 65)
    print("Sample Sort | Distribution:", dist)
    print("=" * 65)
    distribution_runs[dist] = run_mpi(
        n=100000, algorithm="sample", strategy="oversample",
        distribution=dist, processes=4
    )

## 9. Compare all splitter strategies

In [ ]:
strategies = ["regular", "random", "oversample"]
splitter_runs = {}

for strategy in strategies:
    print("\\n" + "=" * 65)
    print("Splitter strategy:", strategy)
    print("=" * 65)
    splitter_runs[strategy] = run_mpi(
        n=100000, algorithm="sample", strategy=strategy,
        distribution="uniform", processes=4
    )

## 10. Compare Sample Sort and Bitonic Sort

In [ ]:
algorithm_runs = {}

for algorithm, strategy in [("sample", "oversample"), ("bitonic", "regular")]:
    print("\\n" + "=" * 65)
    print("Algorithm:", algorithm)
    print("=" * 65)
    algorithm_runs[algorithm] = run_mpi(
        n=100000, algorithm=algorithm, strategy=strategy,
        distribution="uniform", processes=4
    )

## 11. Test MPI process counts (1, 2, 4, 8, 16)

In [ ]:
# Colab may not provide enough CPU resources for high process counts.
# --oversubscribe permits launch, but does not create additional CPU capacity.
process_counts = [1, 2, 4, 8, 16]
scaling_runs = {}

for p in process_counts:
    print("\\n" + "=" * 65)
    print("Sample Sort with", p, "MPI processes")
    print("=" * 65)
    scaling_runs[p] = run_mpi(
        n=100000, algorithm="sample", strategy="oversample",
        distribution="uniform", processes=p
    )

## 10. Execute the real-dataset experiments

We test three dataset sizes (100K, 250K and 500K records) with both algorithms. The same first N records are used for each algorithm at a given size.

**Tip for presentation:** Explain that one MPI rank reads the input file, then `MPI_Scatterv` distributes records. The sorting algorithm performs distributed sorting and the program gathers and verifies the result.


In [ ]:
DATASET_SIZES = [100_000, 250_000, 500_000]
MPI_PROCESSES = 4
real_results = []

for n_records in DATASET_SIZES:
    if n_records > len(retail_keys):
        print(f"Skipping {n_records}: dataset has only {len(retail_keys)} valid records.")
        continue

    input_path = f"/content/retail_unitprice_{n_records}.txt"
    retail_keys.iloc[:n_records].to_csv(input_path, index=False, header=False)

    print("\n" + "=" * 70)
    print(f"UCI Online Retail | {n_records:,} records")
    print("=" * 70)

    for algorithm, strategy in [("sample", "oversample"), ("bitonic", "regular")]:
        print(f"\nRunning {algorithm} with {MPI_PROCESSES} MPI processes")
        result = run_mpi_file(
            input_path=input_path,
            algorithm=algorithm,
            strategy=strategy,
            processes=MPI_PROCESSES
        )
        real_results.append(
            parse_file_metrics(result, algorithm, MPI_PROCESSES, n_records, strategy)
        )

real_results_df = pd.DataFrame(real_results)
display(real_results_df)
real_results_df.to_csv("/content/team22_uci_retail_results.csv", index=False)
print("Saved: /content/team22_uci_retail_results.csv")


## 11. Real-dataset performance visualizations

The following graphs use only the UCI dataset experiments. They compare algorithms across input sizes and show how execution time, throughput and communication time change as the dataset grows.


In [ ]:
# Convert metrics to numeric values for plotting
for column in ["Total Time (s)", "Keys Per Second", "Communication Time (s)"]:
    real_results_df[column] = pd.to_numeric(real_results_df[column], errors="coerce")

# Graph 1: execution time by algorithm and dataset size
pivot_time = real_results_df.pivot(index="Records", columns="Algorithm", values="Total Time (s)")
ax = pivot_time.plot(kind="bar", figsize=(9, 5))
ax.set_title("UCI Online Retail: Execution Time by Dataset Size")
ax.set_xlabel("Number of records")
ax.set_ylabel("Execution time (seconds)")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y")
plt.tight_layout()
plt.show()

# Graph 2: throughput
pivot_rate = real_results_df.pivot(index="Records", columns="Algorithm", values="Keys Per Second") / 1e6
ax = pivot_rate.plot(marker="o", figsize=(9, 5))
ax.set_title("UCI Online Retail: Throughput Comparison")
ax.set_xlabel("Number of records")
ax.set_ylabel("Million records per second")
ax.grid(True)
plt.tight_layout()
plt.show()

# Graph 3: communication time
pivot_comm = real_results_df.pivot(index="Records", columns="Algorithm", values="Communication Time (s)")
ax = pivot_comm.plot(kind="bar", figsize=(9, 5))
ax.set_title("UCI Online Retail: Communication Time")
ax.set_xlabel("Number of records")
ax.set_ylabel("Communication time (seconds)")
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y")
plt.tight_layout()
plt.show()

# Graph 4: correctness summary
correct_counts = real_results_df["Correctness"].value_counts()
plt.figure(figsize=(6, 4))
plt.bar(correct_counts.index, correct_counts.values)
plt.title("Correctness of Real-Dataset Sorting Runs")
plt.xlabel("Validation result")
plt.ylabel("Number of experiments")
plt.grid(axis="y")
plt.show()


## 12. Explain the real-dataset results

Use the generated table and graphs to answer these questions:

1. Which algorithm has the lower execution time for each dataset size?
2. Does throughput increase or decrease as the input grows?
3. How much time is spent communicating between MPI processes?
4. Did every run pass the sorted-order and record-preservation checks?
5. Does the best algorithm change with dataset size?

**Interpretation rule:** Do not assume the algorithm with the lowest time in one run is universally faster. Repeat experiments and compare medians if you need a stronger performance claim. Timing components may overlap depending on how the C++ program measures them, so do not add them together unless the implementation guarantees they are disjoint.


In [ ]:
# Download the real-dataset experiment results
from google.colab import files
files.download("/content/team22_uci_retail_results.csv")


## 12. Parse experiment output and build a results table

In [ ]:
import re
import pandas as pd
import numpy as np

def parse_metrics(result, algorithm, processes, distribution, strategy):
    output = result.stdout if result else ""

    def extract(pattern):
        match = re.search(pattern, output)
        return float(match.group(1)) if match else np.nan

    correctness = re.search(r"Correctness:\s*(PASS|FAIL)", output)

    return {
        "Algorithm": algorithm,
        "Processes": processes,
        "Distribution": distribution,
        "Splitter": strategy if algorithm == "sample" else "-",
        "Correctness": correctness.group(1) if correctness else "ERROR",
        "Total Time (s)": extract(r"Total time \(max rank, s\):\s*([0-9.eE+-]+)"),
        "Local Sort Time (s)": extract(r"Local sort time \(max rank, s\):\s*([0-9.eE+-]+)"),
        "Communication Time (s)": extract(r"Communication time \(max rank, s\):\s*([0-9.eE+-]+)"),
        "Merge Time (s)": extract(r"Merge/final-sort time \(max rank, s\):\s*([0-9.eE+-]+)"),
        "Keys Per Second": extract(r"Keys/second:\s*([0-9.eE+-]+)"),
        "Load Imbalance": extract(r"Load imbalance \(max rank keys / average keys\):\s*([0-9.eE+-]+)")
    }

rows = []

for dist, result in distribution_runs.items():
    rows.append(parse_metrics(result, "sample", 4, dist, "oversample"))

for strategy, result in splitter_runs.items():
    rows.append(parse_metrics(result, "sample", 4, "uniform", strategy))

for algorithm, result in algorithm_runs.items():
    rows.append(parse_metrics(
        result, algorithm, 4, "uniform",
        "oversample" if algorithm == "sample" else "regular"
    ))

for p, result in scaling_runs.items():
    rows.append(parse_metrics(result, "sample", p, "uniform", "oversample"))

results_df = pd.DataFrame(rows)
results_df = results_df.drop_duplicates(
    subset=["Algorithm", "Processes", "Distribution", "Splitter"],
    keep="last"
).reset_index(drop=True)

display(results_df)
results_df.to_csv("/content/team22_experimental_results.csv", index=False)
print("Saved current synthetic benchmark results: /content/team22_experimental_results.csv")
print("\nCorrectness summary:")
print(results_df["Correctness"].value_counts(dropna=False))


## 13. Generate performance graphs

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

CSV_PATH = "/content/team22_experimental_results.csv"

df = pd.read_csv(CSV_PATH)

print("Dataset loaded successfully!")
print("Total experiments:", len(df))
display(df.head())

**Title: Comparison of Sorting Algorithms**

Objective: To compare the execution performance of Sample Sort and Bitonic Sort using four MPI processes on a uniformly distributed dataset containing 100,000 keys.

Analysis: The graph compares the total execution time of both algorithms. Sample Sort recorded an execution time of approximately 0.023 seconds, whereas Bitonic Sort recorded approximately 0.024 seconds.

Observation: Sample Sort performed slightly faster than Bitonic Sort in this experiment. Bitonic Sort, however, achieved perfectly balanced output distribution across all four processes.

Inference: Sample Sort demonstrated a small performance advantage for the tested dataset, while Bitonic Sort demonstrated better workload balance.

In [ ]:
comparison = df[
    (df["Processes"] == 4) &
    (df["Distribution"] == "uniform") &
    (df["Splitter"].isin(["oversample", "-"]))
].copy()

comparison = comparison.drop_duplicates(subset=["Algorithm"], keep="first")

plt.figure(figsize=(9, 5))
plt.bar(comparison["Algorithm"], comparison["Total Time (s)"])
plt.title("Sample Sort vs Bitonic Sort")
plt.xlabel("Sorting Algorithm")
plt.ylabel("Execution Time (seconds)")
plt.grid(axis="y")
plt.show()

**Title: Impact of MPI Process Count on Execution Time**

Objective: To analyse how increasing the number of MPI processes affects the execution time of distributed Sample Sort.

Analysis: The experiment was conducted using 1, 2, 4, 8 and 16 MPI processes, with 100,000 uniformly distributed keys.

Observation: The lowest execution time was recorded with one process (0.0174 seconds). Execution time increased to approximately 0.0275 seconds with 16 processes.

Inference: Increasing the number of processes does not always improve performance. Communication overhead and process coordination can outweigh the benefits of parallel execution, particularly when the available CPU resources are limited.

In [ ]:
scaling = df[
    (df["Algorithm"] == "sample") &
    (df["Distribution"] == "uniform") &
    (df["Splitter"] == "oversample")
].sort_values("Processes").drop_duplicates(
    subset=["Processes"], keep="last"
)

plt.figure(figsize=(9, 5))
plt.plot(scaling["Processes"], scaling["Total Time (s)"], marker="o")
plt.title("MPI Process Count vs Execution Time")
plt.xlabel("Number of MPI Processes")
plt.ylabel("Execution Time (seconds)")
plt.xticks(scaling["Processes"])
plt.grid(True)
plt.show()

**Title: Throughput vs Number of MPI Processes**

Objective: To measure the processing capacity of the distributed sorting system for different MPI process counts.

Analysis: Throughput represents the number of keys processed per second. It is calculated by dividing the number of input keys by total execution time.

Observation: The highest throughput was approximately 5.74 million keys per second with one process. The throughput decreased to approximately 3.63 million keys per second with 16 processes.

Inference: The results indicate that increasing the number of processes did not improve throughput for this dataset and execution environment. Communication and scheduling overhead affected the overall processing capacity.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    scaling["Processes"],
    scaling["Keys Per Second"] / 1e6,
    marker="o"
)
plt.title("Throughput vs MPI Process Count")
plt.xlabel("Number of MPI Processes")
plt.ylabel("Million Keys per Second")
plt.xticks(scaling["Processes"])
plt.grid(True)
plt.show()

**Title: Performance Evaluation of Splitter Strategies**

Objective: To compare Regular, Random and Oversampling splitter strategies in Sample Sort based on workload distribution.

Analysis: The load imbalance ratio measures how evenly keys are distributed among MPI processes. A value closer to 1 indicates better workload balance.

Observation:
- Regular splitter: 1.341
- Random splitter: 1.528
- Oversampling splitter: 1.154
Oversampling produced the lowest load imbalance among the three strategies.

Inference: Oversampling provided better workload distribution for the tested uniform dataset. Balanced distribution helps prevent individual processes from becoming bottlenecks during parallel sorting.

In [ ]:
splitters = df[
    (df["Algorithm"] == "sample") &
    (df["Processes"] == 4) &
    (df["Distribution"] == "uniform") &
    (df["Splitter"].isin(["regular", "random", "oversample"]))
]

plt.figure(figsize=(9, 5))
plt.bar(splitters["Splitter"], splitters["Load Imbalance"])
plt.axhline(1.0, linestyle="--", label="Perfect balance")
plt.title("Load Imbalance by Splitter Strategy")
plt.xlabel("Splitter Strategy")
plt.ylabel("Load Imbalance Ratio")
plt.legend()
plt.grid(axis="y")
plt.show()

**Title: Effect of Input Distribution on Sorting Performance**

Objective: To evaluate the behaviour of Sample Sort under different input distributions: Uniform, Gaussian, Sorted, Reverse and Duplicate.

Analysis: All five distributions were tested using four MPI processes and the oversampling splitter strategy.

Observation:
- Uniform distribution: Correctness PASS.
- Gaussian distribution: Correctness PASS.
- Sorted distribution: Correctness FAIL.
- Reverse distribution: Correctness FAIL.
- Duplicate distribution: Correctness PASS.
The execution times varied across the input distributions.

Inference: Input distribution influences sorting behaviour and performance. The failures for sorted and reverse inputs indicate that the current implementation requires further correctness testing and debugging before it can be considered reliable for all input types.

In [ ]:
distribution = df[
    (df["Algorithm"] == "sample") &
    (df["Processes"] == 4) &
    (df["Splitter"] == "oversample")
].drop_duplicates(subset=["Distribution"], keep="first")

plt.figure(figsize=(10, 5))
plt.bar(
    distribution["Distribution"],
    distribution["Total Time (s)"]
)
plt.title("Sorting Performance Across Input Distributions")
plt.xlabel("Input Distribution")
plt.ylabel("Execution Time (seconds)")
plt.grid(axis="y")
plt.show()

**Title: Correctness Validation of Distributed Sorting**

Objective: To verify whether the distributed sorting algorithms produce correctly sorted output across all experimental configurations.

Analysis: Each experiment includes a correctness check that determines whether the resulting output satisfies the sorting requirements.

Observation: Out of 13 recorded experiments, 11 passed and 2 failed. Both failures occurred with Sample Sort on sorted and reverse input distributions.

Inference: The majority of experiments produced correct results. However, the failures demonstrate the importance of validating distributed sorting algorithms across different input patterns rather than testing only random or uniformly distributed data.

In [ ]:
correctness = df["Correctness"].value_counts()

plt.figure(figsize=(7, 5))
plt.pie(
    correctness,
    labels=correctness.index,
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Sorting Correctness Across Experiments")
plt.show()

## Overall Performance Analysis and Conclusion

This project evaluates distributed sorting using Sample Sort and Parallel Bitonic Sort.

The experiments include two complementary forms of testing:

- **Controlled synthetic inputs:** uniform, Gaussian, sorted, reverse and duplicate-key distributions. These help isolate the effect of input patterns and splitter strategies.
- **Real-world input:** UCI Online Retail transaction data, using `UnitPrice` converted into integer pence. This demonstrates sorting on values derived from a practical dataset.

The evaluation considers correctness, total execution time, local sorting time, communication time, throughput and load imbalance.

**Conclusion:** Distributed sorting performance depends on the algorithm, data distribution, number of processes and communication overhead. Parallel execution does not automatically guarantee faster results, especially when the number of MPI processes exceeds the CPU resources available in the Colab runtime. Correctness validation is essential for every input type and dataset size.


In [ ]:
import os

GRAPH_DIR = "/content/team22_graphs"
os.makedirs(GRAPH_DIR, exist_ok=True)

def save_graph(filename):
    plt.tight_layout()
    plt.savefig(
        os.path.join(GRAPH_DIR, filename),
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

# Execution time by process count
plt.figure(figsize=(9, 5))
plt.plot(scaling["Processes"], scaling["Total Time (s)"], marker="o")
plt.title("MPI Scalability")
plt.xlabel("MPI Processes")
plt.ylabel("Execution Time (s)")
plt.xticks(scaling["Processes"])
plt.grid(True)
save_graph("01_mpi_scalability.png")

# Throughput
plt.figure(figsize=(9, 5))
plt.plot(scaling["Processes"], scaling["Keys Per Second"] / 1e6, marker="o")
plt.title("Throughput Analysis")
plt.xlabel("MPI Processes")
plt.ylabel("Million Keys/Second")
plt.xticks(scaling["Processes"])
plt.grid(True)
save_graph("02_throughput.png")

# Splitter comparison
plt.figure(figsize=(9, 5))
plt.bar(splitters["Splitter"], splitters["Load Imbalance"])
plt.axhline(1.0, linestyle="--", label="Perfect balance")
plt.title("Splitter Load Balancing")
plt.xlabel("Splitter Strategy")
plt.ylabel("Load Imbalance")
plt.legend()
save_graph("03_splitter_comparison.png")

print("All graphs saved in:", GRAPH_DIR)

## 14. Download CSV results

In [ ]:
from google.colab import files
files.download("/content/team22_experimental_results.csv")

## Download source and outputs

The C++ MPI source is generated in `/content/team22_distributed_sort.cpp`. The real-dataset results are saved as `/content/team22_uci_retail_results.csv`.

To download the source if needed:

```python
from google.colab import files
files.download(CPP_FILE)
```

**Presentation note:** Explain that UnitPrice is converted to integer pence, and each input record receives a unique ID. The ID lets the program verify that sorting has not lost or duplicated records.
